# Day 7 — Additional Models + Evaluation (MVP)

**Goal:** benchmark multiple classifiers with **CV on train** and a final **test** table; save the current best model.

> **Read first:** [CONCEPTS.md](./CONCEPTS.md)


## Setup

Run with cwd `notebooks/day07/`. Benchmark logic: `src/model_evaluation.py`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "dataset").is_dir() and (p / "requirements.txt").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model_evaluation import benchmark_classifiers, pick_best_model_name, save_day07_artifacts
from src.preprocessing import load_cleaned_table, prepare_train_test_bundle, save_artifacts

sns.set_theme(style="whitegrid", context="notebook")

## 1. Load features (Day 3 pipeline)

In [ ]:
df = load_cleaned_table(PROJECT_ROOT)
bundle = prepare_train_test_bundle(df, test_size=0.2, random_state=42)
save_artifacts(bundle, PROJECT_ROOT)

X_train, X_test = bundle["X_train"], bundle["X_test"]
y_train, y_test = bundle["y_train"], bundle["y_test"]
class_names = bundle["career_classes"]

## 2. Benchmark: 5-fold CV on train + test metrics

Models: logistic, random forest, SVM (RBF), KNN (k=7), and **XGBoost** if `pip install xgboost` was done.

In [ ]:
comparison, fitted, test_evals = benchmark_classifiers(
    X_train,
    X_test,
    y_train,
    y_test,
    class_names,
    cv=5,
)
comparison

## 3. Select best model (by test macro F1)

In [ ]:
best_name = pick_best_model_name(comparison)
print("Best model:", best_name)
comparison.loc[comparison["model"] == best_name]

## 4. Classification report for the winner (test set)

In [ ]:
print(test_evals[best_name]["classification_report"])

## 5. Save benchmark table + best model (MVP artifact)

In [ ]:
paths = save_day07_artifacts(comparison, fitted, class_names, PROJECT_ROOT, best_name=best_name)
paths

## Milestone checklist

- [ ] `outputs/metrics/day07_model_benchmark.csv` contains **your** run
- [ ] You can explain why the best model won (macro F1, not accuracy alone)
- [ ] End-to-end path: profile → preprocess → predict → evaluate

**Before Day 8:** keep `models/day07_best_classifier.joblib` — neural nets will compare against this.